In [1]:
import pickle
import pandas as pd

In [2]:
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/imb_files/dt_imb_152_entire.pkl', 'rb') as f:
    xgb_results = pickle.load(f)

## Baseline

In [3]:
mean_accuracy = []
std_accuracy = []
mean_recall_void = []
std_recall_void = []
for win_data, data in xgb_results.items(): 
    data_ = data["none"]
    
    mean_accuracy.append(data_['mean_accuracy'])
    std_accuracy.append(data_['std_accuracy'])
    mean_recall_void.append(data_['mean_recall_minority'])
    std_recall_void.append(data_['std_recall_minority'])

indexes = xgb_results.keys()
results = {
    'config': indexes,
    'mean_accuracy': mean_accuracy,
    'std_accuracy': std_accuracy,
    'mean_recall_void': mean_recall_void,
    'std_recall_void': std_recall_void
} 
    
df = pd.DataFrame(results)
# df = pd.DataFrame(results, index=indexes)


In [4]:
df 

,config,mean_accuracy,std_accuracy,mean_recall_void,std_recall_void
0,1s_no,0.594265,0.056649,0.490753,0.079245
1,1s_0.5,0.621583,0.075713,0.542094,0.105503
2,1s_0.8,0.612894,0.057760,0.495909,0.084471
3,2s_no,0.633163,0.086425,0.543402,0.153000
4,2s_0.5,0.624053,0.081730,0.532294,0.140227
5,2s_0.8,0.631851,0.068343,0.534889,0.120684
6,3s_no,0.680515,0.098133,0.575791,0.087302
7,3s_0.5,0.605548,0.106440,0.530716,0.146976
8,3s_0.8,0.631695,0.093867,0.547779,0.131477
9,4s_no,0.610364,0.067941,0.523841,0.166469


In [8]:
# Get indexes of top N values using nlargest()
def get_top_n_indexes(df, column, n=5):
    """Get indexes of top N values in specified column"""
    top_indexes = df.nlargest(n, column).index.tolist()
    return top_indexes

# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_recall_void', 5)
print(f"\nTop 5 indexes by mean_recall_void:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_recall_void']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Get the std of the top 5 recall values
print("\nCorresponding std of top recall values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'std_recall_void']
    print(f"  Index {idx}: {config} = {value:.6f}")


Top 5 indexes by mean_recall_void:
Indexes: [13, 6, 11, 8, 3]
Corresponding configurations and values:
  Index 13: 5s_0.5 = 0.599084
  Index 6: 3s_no = 0.575791
  Index 11: 4s_0.8 = 0.572216
  Index 8: 3s_0.8 = 0.547779
  Index 3: 2s_no = 0.543402

Corresponding std of top recall values:
  Index 13: 5s_0.5 = 0.139506
  Index 6: 3s_no = 0.087302
  Index 11: 4s_0.8 = 0.103912
  Index 8: 3s_0.8 = 0.131477
  Index 3: 2s_no = 0.153000


In [9]:
# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_accuracy', 5)
print(f"\nTop 5 indexes by mean_accuracy:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")
    
# Get the std of the top 5 recall values
print("\nCorresponding std of top recall values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'std_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")


Top 5 indexes by mean_accuracy:
Indexes: [6, 13, 11, 14, 3]
Corresponding configurations and values:
  Index 6: 3s_no = 0.680515
  Index 13: 5s_0.5 = 0.662525
  Index 11: 4s_0.8 = 0.636335
  Index 14: 5s_0.8 = 0.634495
  Index 3: 2s_no = 0.633163

Corresponding std of top recall values:
  Index 6: 3s_no = 0.098133
  Index 13: 5s_0.5 = 0.086062
  Index 11: 4s_0.8 = 0.101826
  Index 14: 5s_0.8 = 0.098902
  Index 3: 2s_no = 0.086425


DF of void recall and accuracy values together

In [10]:
# Function to create results as DataFrames
def analyze_top_values_as_dataframes(df, column='mean_recall_void', std_column='std_recall_void', n=5):
    """
    Analyze DataFrame and return results as DataFrames
    
    Returns:
        dict: Dictionary containing different analysis results as DataFrames
    """
    results = {}
    
    # 1. Top 5 void recall values DataFrame
    top_n_indexes = get_top_n_indexes(df, "mean_recall_void", n)
    top_5_vr_df = df.loc[top_n_indexes, ['config', column, std_column]].copy() # top 5 void recall
    top_5_vr_df = top_5_vr_df.reset_index()
    top_5_vr_df = top_5_vr_df.rename(columns={"config": "vr_win_os"})
    top_5_vr_df['rank'] = range(1, len(top_5_vr_df) + 1)
    top_5_vr_df = top_5_vr_df[['rank', 'vr_win_os', column, std_column]]
    # print(top_5_vr_df)
    
    # 2. Top 5 accuracy values DataFrame
    top_n_indexes = get_top_n_indexes(df, "mean_accuracy", n)
    top_5_acc__df = df.loc[top_n_indexes, ['config', "mean_accuracy", "std_accuracy"]].copy() # top 5 accuracy
    top_5_acc__df = top_5_acc__df.rename(columns={"config": "acc_win_os"})
    top_5_acc__df['rank'] = range(1, len(top_5_acc__df) + 1)
    # print(top_5_acc__df)

    
    results = pd.merge(top_5_vr_df,top_5_acc__df)
    return results

# Example usage:
results = analyze_top_values_as_dataframes(df)
results



,rank,vr_win_os,mean_recall_void,std_recall_void,acc_win_os,mean_accuracy,std_accuracy
0,1,5s_0.5,0.599084,0.139506,3s_no,0.680515,0.098133
1,2,3s_no,0.575791,0.087302,5s_0.5,0.662525,0.086062
2,3,4s_0.8,0.572216,0.103912,4s_0.8,0.636335,0.101826
3,4,3s_0.8,0.547779,0.131477,5s_0.8,0.634495,0.098902
4,5,2s_no,0.543402,0.153000,2s_no,0.633163,0.086425


## Oversample

In [3]:
mean_accuracy = []
std_accuracy = []
mean_recall_void = []
std_recall_void = []
for win_data, data in xgb_results.items(): 
    data_ = data["oversample_smote"]
    
    mean_accuracy.append(data_['mean_accuracy'])
    std_accuracy.append(data_['std_accuracy'])
    mean_recall_void.append(data_['mean_recall_minority'])
    std_recall_void.append(data_['std_recall_minority'])

indexes = xgb_results.keys()
results = {
    'config': indexes,
    'mean_accuracy': mean_accuracy,
    'std_accuracy': std_accuracy,
    'mean_recall_void': mean_recall_void,
    'std_recall_void': std_recall_void
} 
    
df = pd.DataFrame(results)

In [4]:
df

,config,mean_accuracy,std_accuracy,mean_recall_void,std_recall_void
0,1s_no,0.584937,0.065059,0.515526,0.093115
1,1s_0.5,0.630133,0.083721,0.565681,0.136870
2,1s_0.8,0.612503,0.072216,0.512447,0.098037
3,2s_no,0.632787,0.069961,0.558553,0.096219
4,2s_0.5,0.632743,0.078023,0.574245,0.103248
5,2s_0.8,0.624758,0.072824,0.545931,0.077061
6,3s_no,0.666969,0.072602,0.596270,0.126958
7,3s_0.5,0.635325,0.097705,0.567431,0.106531
8,3s_0.8,0.628659,0.089164,0.544629,0.102797
9,4s_no,0.585544,0.110884,0.498011,0.179276


In [5]:
# Get indexes of top N values using nlargest()
def get_top_n_indexes(df, column, n=5):
    """Get indexes of top N values in specified column"""
    top_indexes = df.nlargest(n, column).index.tolist()
    return top_indexes

# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_recall_void', 5)
print(f"\nTop 5 indexes by mean_recall_void:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_recall_void']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Get the std of the top 5 recall values
print("\nCorresponding std of top recall values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'std_recall_void']
    print(f"  Index {idx}: {config} = {value:.6f}")




Top 5 indexes by mean_recall_void:
Indexes: [6, 13, 4, 12, 7]
Corresponding configurations and values:
  Index 6: 3s_no = 0.596270
  Index 13: 5s_0.5 = 0.578853
  Index 4: 2s_0.5 = 0.574245
  Index 12: 5s_no = 0.567821
  Index 7: 3s_0.5 = 0.567431

Corresponding std of top recall values:
  Index 6: 3s_no = 0.126958
  Index 13: 5s_0.5 = 0.156781
  Index 4: 2s_0.5 = 0.103248
  Index 12: 5s_no = 0.163428
  Index 7: 3s_0.5 = 0.106531


In [6]:
# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_accuracy', 5)
print(f"\nTop 5 indexes by mean_accuracy:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")
    
# Get the std of the top 5 recall values
print("\nCorresponding std of top recall values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'std_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")


Top 5 indexes by mean_accuracy:
Indexes: [6, 13, 7, 3, 4]
Corresponding configurations and values:
  Index 6: 3s_no = 0.666969
  Index 13: 5s_0.5 = 0.638639
  Index 7: 3s_0.5 = 0.635325
  Index 3: 2s_no = 0.632787
  Index 4: 2s_0.5 = 0.632743

Corresponding std of top recall values:
  Index 6: 3s_no = 0.072602
  Index 13: 5s_0.5 = 0.109580
  Index 7: 3s_0.5 = 0.097705
  Index 3: 2s_no = 0.069961
  Index 4: 2s_0.5 = 0.078023


Get the df for easy input into excel

In [7]:
# Function to create results as DataFrames
def analyze_top_values_as_dataframes(df, column='mean_recall_void', std_column='std_recall_void', n=5):
    """
    Analyze DataFrame and return results as DataFrames
    
    Returns:
        dict: Dictionary containing different analysis results as DataFrames
    """
    results = {}
    
    # 1. Top 5 void recall values DataFrame
    top_n_indexes = get_top_n_indexes(df, "mean_recall_void", n)
    top_5_vr_df = df.loc[top_n_indexes, ['config', column, std_column]].copy() # top 5 void recall
    top_5_vr_df = top_5_vr_df.reset_index()
    top_5_vr_df = top_5_vr_df.rename(columns={"config": "vr_win_os"})
    top_5_vr_df['rank'] = range(1, len(top_5_vr_df) + 1)
    top_5_vr_df = top_5_vr_df[['rank', 'vr_win_os', column, std_column]]
    # print(top_5_vr_df)
    
    # 2. Top 5 accuracy values DataFrame
    top_n_indexes = get_top_n_indexes(df, "mean_accuracy", n)
    top_5_acc__df = df.loc[top_n_indexes, ['config', "mean_accuracy", "std_accuracy"]].copy() # top 5 accuracy
    top_5_acc__df = top_5_acc__df.rename(columns={"config": "acc_win_os"})
    top_5_acc__df['rank'] = range(1, len(top_5_acc__df) + 1)
    # print(top_5_acc__df)

    
    results = pd.merge(top_5_vr_df,top_5_acc__df)
    return results

# Example usage:
results = analyze_top_values_as_dataframes(df)
results

,rank,vr_win_os,mean_recall_void,std_recall_void,acc_win_os,mean_accuracy,std_accuracy
0,1,3s_no,0.596270,0.126958,3s_no,0.666969,0.072602
1,2,5s_0.5,0.578853,0.156781,5s_0.5,0.638639,0.109580
2,3,2s_0.5,0.574245,0.103248,3s_0.5,0.635325,0.097705
3,4,5s_no,0.567821,0.163428,2s_no,0.632787,0.069961
4,5,3s_0.5,0.567431,0.106531,2s_0.5,0.632743,0.078023


## Undersample

In [11]:
mean_accuracy = []
std_accuracy = []
mean_recall_void = []
std_recall_void = []
for win_data, data in xgb_results.items(): 
    data_ = data["undersample_tomek_links"]
    
    mean_accuracy.append(data_['mean_accuracy'])
    std_accuracy.append(data_['std_accuracy'])
    mean_recall_void.append(data_['mean_recall_minority'])
    std_recall_void.append(data_['std_recall_minority'])

indexes = xgb_results.keys()
results = {
    'config': indexes,
    'mean_accuracy': mean_accuracy,
    'std_accuracy': std_accuracy,
    'mean_recall_void': mean_recall_void,
    'std_recall_void': std_recall_void
} 
    
df = pd.DataFrame(results)

In [12]:
df

,config,mean_accuracy,std_accuracy,mean_recall_void,std_recall_void
0,1s_no,0.619285,0.053653,0.595034,0.069942
1,1s_0.5,0.632689,0.085724,0.585352,0.140062
2,1s_0.8,0.616183,0.056895,0.516983,0.112396
3,2s_no,0.614175,0.094505,0.560530,0.164174
4,2s_0.5,0.588895,0.077040,0.465388,0.099833
5,2s_0.8,0.626609,0.062025,0.546293,0.100850
6,3s_no,0.645496,0.115119,0.581107,0.207219
7,3s_0.5,0.619659,0.074489,0.584779,0.138437
8,3s_0.8,0.638709,0.109178,0.577508,0.186580
9,4s_no,0.583029,0.070519,0.534461,0.154405


In [ ]:
# Get indexes of top N values using nlargest()
def get_top_n_indexes(df, column, n=5):
    """Get indexes of top N values in specified column"""
    top_indexes = df.nlargest(n, column).index.tolist()
    return top_indexes

# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_recall_void', 5)
print(f"\nTop 5 indexes by mean_recall_void:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_recall_void']
    print(f"  Index {idx}: {config} = {value:.6f}")



Top 5 indexes by mean_recall_void:
Indexes: [12, 13, 0, 1, 7]
Corresponding configurations and values:
  Index 12: 5s_no = 0.627340
  Index 13: 5s_0.5 = 0.619535
  Index 0: 1s_no = 0.595034
  Index 1: 1s_0.5 = 0.585352
  Index 7: 3s_0.5 = 0.584779

All indexes sorted by mean_recall_void (highest to lowest):
Indexes: [12, 13, 0, 1, 7, 6, 14, 8, 10, 3, 5, 9, 11, 2, 4]

Indexes with mean_recall_void > 0.55:
Indexes: [0, 1, 3, 6, 7, 8, 10, 12, 13, 14]
Corresponding configurations and values:
  Index 0: 1s_no = 0.595034
  Index 1: 1s_0.5 = 0.585352
  Index 3: 2s_no = 0.560530
  Index 6: 3s_no = 0.581107
  Index 7: 3s_0.5 = 0.584779
  Index 8: 3s_0.8 = 0.577508
  Index 10: 4s_0.5 = 0.569512
  Index 12: 5s_no = 0.627340
  Index 13: 5s_0.5 = 0.619535
  Index 14: 5s_0.8 = 0.579840

Highest mean_recall_void:
Index 12: 5s_no = 0.627340


In [14]:
# Get indexes of top N values using nlargest()
def get_top_n_indexes(df, column, n=5):
    """Get indexes of top N values in specified column"""
    top_indexes = df.nlargest(n, column).index.tolist()
    return top_indexes

# Get indexes using argsort() for all values (sorted)
def get_all_indexes_sorted(df, column):
    """Get all indexes sorted by column values (highest to lowest)"""
    sorted_indexes = df[column].argsort()[::-1].tolist()
    return sorted_indexes

# Get indexes above a certain threshold
def get_indexes_above_threshold(df, column, threshold):
    """Get indexes where column values are above threshold"""
    above_threshold_indexes = df[df[column] > threshold].index.tolist()
    return above_threshold_indexes


# print("DataFrame:")
# print(df)
# print("\n" + "="*50)

# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_accuracy', 5)
print(f"\nTop 5 indexes by mean_accuracy:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Get all indexes sorted
all_sorted_indexes = get_all_indexes_sorted(df, 'mean_accuracy')
print(f"\nAll indexes sorted by mean_accuracy (highest to lowest):")
print(f"Indexes: {all_sorted_indexes}")

# Get indexes above threshold (e.g., 0.55)
threshold = 0.55
above_threshold = get_indexes_above_threshold(df, 'mean_accuracy', threshold)
print(f"\nIndexes with mean_accuracy > {threshold}:")
print(f"Indexes: {above_threshold}")
print("Corresponding configurations and values:")
for idx in above_threshold:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Highest value index
max_index = df['mean_accuracy'].idxmax()
max_value = df.loc[max_index, 'mean_accuracy']
max_config = df.loc[max_index, 'config']
print(f"\nHighest mean_accuracy:")
print(f"Index {max_index}: {max_config} = {max_value:.6f}")


Top 5 indexes by mean_accuracy:
Indexes: [6, 12, 8, 13, 1]
Corresponding configurations and values:
  Index 6: 3s_no = 0.645496
  Index 12: 5s_no = 0.639074
  Index 8: 3s_0.8 = 0.638709
  Index 13: 5s_0.5 = 0.633084
  Index 1: 1s_0.5 = 0.632689

All indexes sorted by mean_accuracy (highest to lowest):
Indexes: [6, 12, 8, 13, 1, 5, 14, 7, 0, 10, 2, 3, 11, 4, 9]

Indexes with mean_accuracy > 0.55:
Indexes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Corresponding configurations and values:
  Index 0: 1s_no = 0.619285
  Index 1: 1s_0.5 = 0.632689
  Index 2: 1s_0.8 = 0.616183
  Index 3: 2s_no = 0.614175
  Index 4: 2s_0.5 = 0.588895
  Index 5: 2s_0.8 = 0.626609
  Index 6: 3s_no = 0.645496
  Index 7: 3s_0.5 = 0.619659
  Index 8: 3s_0.8 = 0.638709
  Index 9: 4s_no = 0.583029
  Index 10: 4s_0.5 = 0.617466
  Index 11: 4s_0.8 = 0.613980
  Index 12: 5s_no = 0.639074
  Index 13: 5s_0.5 = 0.633084
  Index 14: 5s_0.8 = 0.622666

Highest mean_accuracy:
Index 6: 3s_no = 0.645496


## Combined Smote Tomek

In [15]:
mean_accuracy = []
std_accuracy = []
mean_recall_void = []
std_recall_void = []
for win_data, data in xgb_results.items(): 
    data_ = data["combined_smote_tomek"]
    
    mean_accuracy.append(data_['mean_accuracy'])
    std_accuracy.append(data_['std_accuracy'])
    mean_recall_void.append(data_['mean_recall_minority'])
    std_recall_void.append(data_['std_recall_minority'])

indexes = xgb_results.keys()
results = {
    'config': indexes,
    'mean_accuracy': mean_accuracy,
    'std_accuracy': std_accuracy,
    'mean_recall_void': mean_recall_void,
    'std_recall_void': std_recall_void
} 
    
df = pd.DataFrame(results)

In [16]:
df

,config,mean_accuracy,std_accuracy,mean_recall_void,std_recall_void
0,1s_no,0.594686,0.066652,0.534760,0.139510
1,1s_0.5,0.624822,0.085305,0.564021,0.133913
2,1s_0.8,0.619295,0.056964,0.503340,0.081684
3,2s_no,0.626271,0.079254,0.544472,0.109596
4,2s_0.5,0.637853,0.063975,0.564006,0.100302
5,2s_0.8,0.615555,0.077079,0.543614,0.123809
6,3s_no,0.678824,0.073054,0.641984,0.112151
7,3s_0.5,0.611697,0.058915,0.560495,0.099185
8,3s_0.8,0.620623,0.096173,0.552306,0.136857
9,4s_no,0.608519,0.105264,0.539554,0.128878


In [17]:
# Get indexes of top N values using nlargest()
def get_top_n_indexes(df, column, n=5):
    """Get indexes of top N values in specified column"""
    top_indexes = df.nlargest(n, column).index.tolist()
    return top_indexes

# Get indexes using argsort() for all values (sorted)
def get_all_indexes_sorted(df, column):
    """Get all indexes sorted by column values (highest to lowest)"""
    sorted_indexes = df[column].argsort()[::-1].tolist()
    return sorted_indexes

# Get indexes above a certain threshold
def get_indexes_above_threshold(df, column, threshold):
    """Get indexes where column values are above threshold"""
    above_threshold_indexes = df[df[column] > threshold].index.tolist()
    return above_threshold_indexes


# print("DataFrame:")
# print(df)
# print("\n" + "="*50)

# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_recall_void', 5)
print(f"\nTop 5 indexes by mean_recall_void:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_recall_void']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Get all indexes sorted
all_sorted_indexes = get_all_indexes_sorted(df, 'mean_recall_void')
print(f"\nAll indexes sorted by mean_recall_void (highest to lowest):")
print(f"Indexes: {all_sorted_indexes}")

# Get indexes above threshold (e.g., 0.55)
threshold = 0.55
above_threshold = get_indexes_above_threshold(df, 'mean_recall_void', threshold)
print(f"\nIndexes with mean_recall_void > {threshold}:")
print(f"Indexes: {above_threshold}")
print("Corresponding configurations and values:")
for idx in above_threshold:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_recall_void']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Highest value index
max_index = df['mean_recall_void'].idxmax()
max_value = df.loc[max_index, 'mean_recall_void']
max_config = df.loc[max_index, 'config']
print(f"\nHighest mean_recall_void:")
print(f"Index {max_index}: {max_config} = {max_value:.6f}")


Top 5 indexes by mean_recall_void:
Indexes: [13, 6, 12, 14, 11]
Corresponding configurations and values:
  Index 13: 5s_0.5 = 0.650387
  Index 6: 3s_no = 0.641984
  Index 12: 5s_no = 0.593621
  Index 14: 5s_0.8 = 0.586933
  Index 11: 4s_0.8 = 0.572972

All indexes sorted by mean_recall_void (highest to lowest):
Indexes: [13, 6, 12, 14, 11, 1, 4, 7, 10, 8, 3, 5, 9, 0, 2]

Indexes with mean_recall_void > 0.55:
Indexes: [1, 4, 6, 7, 8, 10, 11, 12, 13, 14]
Corresponding configurations and values:
  Index 1: 1s_0.5 = 0.564021
  Index 4: 2s_0.5 = 0.564006
  Index 6: 3s_no = 0.641984
  Index 7: 3s_0.5 = 0.560495
  Index 8: 3s_0.8 = 0.552306
  Index 10: 4s_0.5 = 0.559921
  Index 11: 4s_0.8 = 0.572972
  Index 12: 5s_no = 0.593621
  Index 13: 5s_0.5 = 0.650387
  Index 14: 5s_0.8 = 0.586933

Highest mean_recall_void:
Index 13: 5s_0.5 = 0.650387


In [18]:
# Get indexes of top N values using nlargest()
def get_top_n_indexes(df, column, n=5):
    """Get indexes of top N values in specified column"""
    top_indexes = df.nlargest(n, column).index.tolist()
    return top_indexes

# Get indexes using argsort() for all values (sorted)
def get_all_indexes_sorted(df, column):
    """Get all indexes sorted by column values (highest to lowest)"""
    sorted_indexes = df[column].argsort()[::-1].tolist()
    return sorted_indexes

# Get indexes above a certain threshold
def get_indexes_above_threshold(df, column, threshold):
    """Get indexes where column values are above threshold"""
    above_threshold_indexes = df[df[column] > threshold].index.tolist()
    return above_threshold_indexes


# print("DataFrame:")
# print(df)
# print("\n" + "="*50)

# Get top 5 indexes
top_5_indexes = get_top_n_indexes(df, 'mean_accuracy', 5)
print(f"\nTop 5 indexes by mean_accuracy:")
print(f"Indexes: {top_5_indexes}")
print("Corresponding configurations and values:")
for idx in top_5_indexes:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Get all indexes sorted
all_sorted_indexes = get_all_indexes_sorted(df, 'mean_accuracy')
print(f"\nAll indexes sorted by mean_accuracy (highest to lowest):")
print(f"Indexes: {all_sorted_indexes}")

# Get indexes above threshold (e.g., 0.55)
threshold = 0.55
above_threshold = get_indexes_above_threshold(df, 'mean_accuracy', threshold)
print(f"\nIndexes with mean_accuracy > {threshold}:")
print(f"Indexes: {above_threshold}")
print("Corresponding configurations and values:")
for idx in above_threshold:
    config = df.loc[idx, 'config']
    value = df.loc[idx, 'mean_accuracy']
    print(f"  Index {idx}: {config} = {value:.6f}")

# Highest value index
max_index = df['mean_accuracy'].idxmax()
max_value = df.loc[max_index, 'mean_accuracy']
max_config = df.loc[max_index, 'config']
print(f"\nHighest mean_accuracy:")
print(f"Index {max_index}: {max_config} = {max_value:.6f}")


Top 5 indexes by mean_accuracy:
Indexes: [6, 12, 13, 11, 4]
Corresponding configurations and values:
  Index 6: 3s_no = 0.678824
  Index 12: 5s_no = 0.663602
  Index 13: 5s_0.5 = 0.661323
  Index 11: 4s_0.8 = 0.643363
  Index 4: 2s_0.5 = 0.637853

All indexes sorted by mean_accuracy (highest to lowest):
Indexes: [6, 12, 13, 11, 4, 14, 10, 3, 1, 8, 2, 5, 7, 9, 0]

Indexes with mean_accuracy > 0.55:
Indexes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Corresponding configurations and values:
  Index 0: 1s_no = 0.594686
  Index 1: 1s_0.5 = 0.624822
  Index 2: 1s_0.8 = 0.619295
  Index 3: 2s_no = 0.626271
  Index 4: 2s_0.5 = 0.637853
  Index 5: 2s_0.8 = 0.615555
  Index 6: 3s_no = 0.678824
  Index 7: 3s_0.5 = 0.611697
  Index 8: 3s_0.8 = 0.620623
  Index 9: 4s_no = 0.608519
  Index 10: 4s_0.5 = 0.626726
  Index 11: 4s_0.8 = 0.643363
  Index 12: 5s_no = 0.663602
  Index 13: 5s_0.5 = 0.661323
  Index 14: 5s_0.8 = 0.637621

Highest mean_accuracy:
Index 6: 3s_no = 0.678824
